# 1. Introduction

## 1.1 Notebook description

Generate a combat report: kills per round for all (weapon, defender) combinations using the wh40k combat rules module.

## 1.2 Common imports

Common library imports used across my notebooks

In [1]:
import pandas as pd
import numpy as np

pd.options.display.max_columns = 100
pd.options.display.max_rows = 100
pd.options.display.width = 400


## 1.3 Project imports

In [2]:
from sf_wh.common.unit_loader import read_unit_rules, read_unit_weapons
from sf_wh.common.combat_rules import get_df_atk_matrix, get_df_atk_report, get_r_to_k

# 2. Load data

In [3]:
df_atk = read_unit_weapons('army1', ruleset='gw')
df_def = read_unit_rules('army1', ruleset='gw')

In [4]:
df_atk

,faction,army,family,type,unit,model,is_melee,name,mode,is_half_range,R,A,H,S,AP,D_fixed,D_n_dice,D_dice_size,rapid_fire,blast,melta,sustained_hits,lethal_hits,dev_w,anti_inf,anti_tank,ignore_cover,rr_hit,rr_wound,bonus_hit,bonus_w,crit_h,crit_w
0,imperium,any,Bolter,Basic,any,any,False,Bolt gun,-,False,24,2,3,4,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6
1,imperium,any,Bolter,Basic,any,any,False,Heavy Bolter,-,False,36,3,3,5,1,2,0,0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6
2,imperium,any,Melta,Special,any,any,False,Melta gun,-,False,12,1,3,9,4,0,1,6,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6
3,imperium,any,Melta,Special,any,any,False,Melta gun,close range,True,12,1,3,9,4,0,1,6,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6
4,imperium,any,Melta,Heavy,any,any,False,Multi Melta,-,False,18,2,3,9,4,0,1,6,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6
5,imperium,any,Melta,Heavy,any,any,False,Multi Melta,close range,True,18,2,3,9,4,0,1,6,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6
6,imperium,any,Plasma,Special,any,any,False,Plasma gun,-,False,24,1,3,7,2,1,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6
7,imperium,any,Plasma,Special,any,any,False,Plasma gun,close range,True,24,1,3,7,2,1,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6
8,imperium,any,Melee,Basic,Intercessor,any,True,Close Combat Weapon,-,False,0,3,3,4,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6


In [5]:
df_def

,faction,army,unit,model,is_inf,n_models,T,SV,SV_invul,W,FNP,minus_hit,minus_w,D_subtract,D_halve,rr_save
0,imperium,Astra Militarum,Guard,NaN,True,1,3,5,NaN,1,NaN,0,0,0,False,False
1,imperium,Aeldari,Guardian,NaN,True,1,3,4,NaN,1,NaN,0,0,0,False,False
2,imperium,Orks,Boy,NaN,True,1,5,5,NaN,1,NaN,0,0,0,False,False
3,imperium,Space Marines,Marine,NaN,True,1,4,3,NaN,2,NaN,0,0,0,False,False
4,imperium,Space Marines,Terminator,NaN,True,1,5,2,4.0,3,NaN,0,0,0,False,False
5,imperium,Space Marines,Impulsor,NaN,True,1,9,3,5.0,11,NaN,0,0,0,False,False


# 3. Attack matrix

In [6]:
df_atk_matrix = get_df_atk_matrix(df_atk, df_def)
df_atk_matrix

,faction,army,family,type,unit,model,is_melee,name,mode,is_half_range,R,A,H,S,AP,D_fixed,D_n_dice,D_dice_size,rapid_fire,blast,melta,sustained_hits,lethal_hits,dev_w,anti_inf,anti_tank,ignore_cover,rr_hit,rr_wound,bonus_hit,bonus_w,crit_h,crit_w,def_army,def_unit,def_model,is_inf,n_models,T,SV,SV_invul,W,FNP,minus_hit,minus_w,D_subtract,D_halve,rr_save
0,imperium,any,Bolter,Basic,any,any,False,Bolt gun,-,False,24,2,3,4,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6,Astra Militarum,Guard,NaN,True,1,3,5,NaN,1,NaN,0,0,0,False,False
1,imperium,any,Bolter,Basic,any,any,False,Bolt gun,-,False,24,2,3,4,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6,Aeldari,Guardian,NaN,True,1,3,4,NaN,1,NaN,0,0,0,False,False
2,imperium,any,Bolter,Basic,any,any,False,Bolt gun,-,False,24,2,3,4,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6,Orks,Boy,NaN,True,1,5,5,NaN,1,NaN,0,0,0,False,False
3,imperium,any,Bolter,Basic,any,any,False,Bolt gun,-,False,24,2,3,4,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6,Space Marines,Marine,NaN,True,1,4,3,NaN,2,NaN,0,0,0,False,False
4,imperium,any,Bolter,Basic,any,any,False,Bolt gun,-,False,24,2,3,4,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6,Space Marines,Terminator,NaN,True,1,5,2,4.0,3,NaN,0,0,0,False,False
5,imperium,any,Bolter,Basic,any,any,False,Bolt gun,-,False,24,2,3,4,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6,Space Marines,Impulsor,NaN,True,1,9,3,5.0,11,NaN,0,0,0,False,False
6,imperium,any,Bolter,Basic,any,any,False,Heavy Bolter,-,False,36,3,3,5,1,2,0,0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6,Astra Militarum,Guard,NaN,True,1,3,5,NaN,1,NaN,0,0,0,False,False
7,imperium,any,Bolter,Basic,any,any,False,Heavy Bolter,-,False,36,3,3,5,1,2,0,0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6,Aeldari,Guardian,NaN,True,1,3,4,NaN,1,NaN,0,0,0,False,False
8,imperium,any,Bolter,Basic,any,any,False,Heavy Bolter,-,False,36,3,3,5,1,2,0,0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6,Orks,Boy,NaN,True,1,5,5,NaN,1,NaN,0,0,0,False,False
9,imperium,any,Bolter,Basic,any,any,False,Heavy Bolter,-,False,36,3,3,5,1,2,0,0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,False,False,False,0,0,6,6,Space Marines,Marine,NaN,True,1,4,3,NaN,2,NaN,0,0,0,False,False


## x.1 tmp -debug

In [7]:
from sf_wh.common.combat_rules import get_prob_h, get_prob_w, get_prob_save

In [8]:
df_prob_h = get_prob_h(**df_atk_matrix)
df_prob_w = get_prob_w(**df_atk_matrix)
df_prob_save = get_prob_save(**df_atk_matrix)

In [9]:
get_df_atk_report(df_atk_matrix, pd.concat([df_prob_h, df_prob_w, df_prob_save], axis=1))

,faction,army,unit,model,name,mode,def_army,def_unit,def_model,net_bonus,H_mod,prob_h,prob_crit_h,div_s_t,prob_w,prob_crit_w,cover_mod,min_save,sv_modified,prob_save
0,imperium,any,any,any,Bolt gun,-,Astra Militarum,Guard,NaN,0,3,0.666667,0.166667,1.333333,0.666667,0.166667,True,3,4,0.500000
1,imperium,any,any,any,Bolt gun,-,Aeldari,Guardian,NaN,0,3,0.666667,0.166667,1.333333,0.666667,0.166667,True,3,3,0.666667
2,imperium,any,any,any,Bolt gun,-,Orks,Boy,NaN,0,3,0.666667,0.166667,0.800000,0.333333,0.166667,True,3,4,0.500000
3,imperium,any,any,any,Bolt gun,-,Space Marines,Marine,NaN,0,3,0.666667,0.166667,1.000000,0.500000,0.166667,True,3,3,0.666667
4,imperium,any,any,any,Bolt gun,-,Space Marines,Terminator,NaN,0,3,0.666667,0.166667,0.800000,0.333333,0.166667,True,2,2,0.833333
5,imperium,any,any,any,Bolt gun,-,Space Marines,Impulsor,NaN,0,3,0.666667,0.166667,0.444444,0.166667,0.166667,True,3,3,0.666667
6,imperium,any,any,any,Heavy Bolter,-,Astra Militarum,Guard,NaN,0,3,0.666667,0.166667,1.666667,0.666667,0.166667,True,3,5,0.333333
7,imperium,any,any,any,Heavy Bolter,-,Aeldari,Guardian,NaN,0,3,0.666667,0.166667,1.666667,0.666667,0.166667,True,3,4,0.500000
8,imperium,any,any,any,Heavy Bolter,-,Orks,Boy,NaN,0,3,0.666667,0.166667,1.000000,0.500000,0.166667,True,3,5,0.333333
9,imperium,any,any,any,Heavy Bolter,-,Space Marines,Marine,NaN,0,3,0.666667,0.166667,1.250000,0.666667,0.166667,True,3,3,0.666667


# 4. Rounds to kill

In [10]:
df_r_to_k = get_r_to_k(df_atk_matrix)
df_r_to_k

0       2.250000
1       3.375000
2       4.500000
3       9.000000
4      40.500000
5     148.500000
6       0.900000
7       1.200000
8       1.200000
9       0.900000
10      4.800000
11     10.800000
12      1.800000
13      1.800000
14      2.250000
15      1.080000
16      1.500000
17      5.142857
18      1.800000
19      1.800000
20      2.250000
21      1.080000
22      1.500000
23      5.142857
24      0.900000
25      0.900000
26      1.125000
27      0.540000
28      0.750000
29      2.571429
30      0.900000
31      0.900000
32      1.125000
33      0.540000
34      0.750000
35      2.571429
36      2.160000
37      2.700000
38      2.700000
39      9.000000
40     20.250000
41     99.000000
42      1.080000
43      1.350000
44      1.350000
45      4.500000
46     10.125000
47     49.500000
48      1.125000
49      1.500000
50      2.250000
51      6.000000
52     27.000000
53     99.000000
dtype: float64

In [11]:
df_r_to_k_report = get_df_atk_report(df_atk_matrix, pd.DataFrame(dict(r_to_k=df_r_to_k)))
df_r_to_k_report.tail()

,faction,army,unit,model,name,mode,def_army,def_unit,def_model,r_to_k
49,imperium,any,Intercessor,any,Close Combat Weapon,-,Aeldari,Guardian,NaN,1.50
50,imperium,any,Intercessor,any,Close Combat Weapon,-,Orks,Boy,NaN,2.25
51,imperium,any,Intercessor,any,Close Combat Weapon,-,Space Marines,Marine,NaN,6.00
52,imperium,any,Intercessor,any,Close Combat Weapon,-,Space Marines,Terminator,NaN,27.00
53,imperium,any,Intercessor,any,Close Combat Weapon,-,Space Marines,Impulsor,NaN,99.00


In [12]:
df_r_to_k_report_wide = (
    df_r_to_k_report
    [['name', 'mode','def_unit', 'r_to_k']]
    .set_index(['name', 'mode','def_unit'])
    .unstack(sort=False)
    .round(3)
)
df_r_to_k_report_wide

r_to_k                                           
def_unit                         Guard Guardian    Boy Marine Terminator Impulsor
name                mode                                                         
Bolt gun            -            2.250    3.375  4.500   9.00     40.500  148.500
Heavy Bolter        -            0.900    1.200  1.200   0.90      4.800   10.800
Melta gun           -            1.800    1.800  2.250   1.08      1.500    5.143
                    close range  1.800    1.800  2.250   1.08      1.500    5.143
Multi Melta         -            0.900    0.900  1.125   0.54      0.750    2.571
                    close range  0.900    0.900  1.125   0.54      0.750    2.571
Plasma gun          -            2.160    2.700  2.700   9.00     20.250   99.000
                    close range  1.080    1.350  1.350   4.50     10.125   49.500
Close Combat Weapon -            1.125    1.500  2.250   6.00     27.000   99.000